# Smicanje i kapilarnost — predvidi, izračunaj, provjeri

**Poglavlje U02: reologija i međupovršinske pojave**

Osnovni kapilarni slučaj koristi etanol i podatke Z4. Zatim provjeravamo
konstitutivni model iz Z5, odvajamo stanja mikrodozatora P4 i Z6 te
uspoređujemo tlačnu rezervu s veličinom zanemarenog učinka.

## 1. Predvidi

1. Hoće li prepolovljen promjer približno udvostručiti visinu uspona?
2. Što se događa pri kontaktnom kutu $90^\circ$, a što iznad njega?
3. Ako želimo uspon od 30 mm, očekuješ li promjer bliži 0,1 mm ili 10 mm?

Zapiši predviđanje prije pokretanja ćelija.

4. Može li jedna mjerna točka dokazati stalnu viskoznost?
5. Hoće li uža igla smanjiti tlak pri već formiranoj kapljici istog promjera?
6. Je li mala rezerva regulatora dovoljna ako je zanemareni učinak veći od nje?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
G = 9.81

def kapilarni_uspon(sigma_N_m, theta_deg, d_m, rho_kg_m3=998.0):
    theta = np.radians(theta_deg)
    return 4.0 * sigma_N_m * np.cos(theta) / (rho_kg_m3 * G * d_m)

sigma, theta, d, rho = 0.022, 18.0, 0.60e-3, 790.0
h = kapilarni_uspon(sigma, theta, d, rho)
print(f"Osnovni scenarij: h = {1000*h:.2f} mm")


## 2. Izračunaj — inverzni problem bisekcijom

Umjesto izravnog uvrštavanja odabiremo ciljani uspon $h^*=30$ mm i
numerički tražimo promjer. Bisekcija zadržava interval u kojem funkcija
$h(d)-h^*$ mijenja predznak; širina intervala daje izravnu procjenu pogreške.


In [ ]:
def promjer_bisekcijom(h_cilj, sigma, theta_deg, rho,
                       d_lijevo=0.02e-3, d_desno=10e-3,
                       tol=1e-10, max_iter=80):
    f = lambda d: kapilarni_uspon(sigma, theta_deg, d, rho) - h_cilj
    a, b = d_lijevo, d_desno
    if f(a) * f(b) >= 0:
        raise ValueError("Početni interval ne omeđuje rješenje.")
    povijest = []
    for _ in range(max_iter):
        c = 0.5 * (a + b)
        if f(a) * f(c) <= 0:
            b = c
        else:
            a = c
        povijest.append(b - a)
        if b - a < tol:
            break
    return 0.5 * (a + b), np.asarray(povijest)

h_cilj = 30e-3
d_star, sirina = promjer_bisekcijom(h_cilj, sigma, theta, rho)
print(f"d* = {1000*d_star:.6f} mm nakon {len(sirina)} iteracija")
print(f"konačna granica pogreške < {0.5*1000*sirina[-1]:.3e} mm")

fig, ax = plt.subplots(figsize=(6.8, 3.8))
ax.semilogy(np.arange(1, len(sirina)+1), 0.5*sirina, "o-")
ax.set(xlabel="iteracija", ylabel="granica pogreške promjera (m)",
       title="Konvergencija bisekcije")
ax.grid(ls=":", alpha=0.6)
plt.show()


## 3. Provjeri — osjetljivost i propagacija nesigurnosti

Nesigurnost rezultata računamo deterministički iz centralnih konačnih
razlika. Tako ista procedura vrijedi i kada formula postane složenija.

Ovo je zasebna nastavna dopuna: `ux` su neovisne standardne nesigurnosti
ulaza, a ne granice zadane u Z4 ili Z6. Derivacija po kutu računa se po stupnju.


In [ ]:
x0 = np.array([sigma, theta, d, rho])
ux = np.array([0.0005, 1.0, 0.01e-3, 1.0])
korak = np.array([1e-6, 1e-3, 1e-8, 1e-3])

def model_x(x):
    return kapilarni_uspon(x[0], x[1], x[2], x[3])

grad = np.empty(4)
for i in range(4):
    xp, xm = x0.copy(), x0.copy()
    xp[i] += korak[i]
    xm[i] -= korak[i]
    grad[i] = (model_x(xp) - model_x(xm)) / (2*korak[i])
doprinosi = np.abs(grad * ux)
u_h = np.sqrt(np.sum(doprinosi**2))
print(f"h = {1000*h:.2f} ± {1000*u_h:.2f} mm")
print("Doprinosi u_h [mm] za sigma, theta, d, rho:",
      np.round(1000*doprinosi, 3))

# Referentno rješenje inverznog problema i dva granična/scaling testa.
assert abs(kapilarni_uspon(sigma, theta, d_star, rho) - h_cilj) < 1e-8
assert abs(kapilarni_uspon(sigma, 90.0, d, rho)) < 1e-14
assert np.isclose(kapilarni_uspon(sigma, theta, d/2, rho), 2*h,
                  rtol=1e-12)
print("PASS: inverzno rješenje, kut 90° i skaliranje 1/d su potvrđeni.")

# Neovisna analitička provjera numeričkih derivacija i inverznog promjera.
grad_ref = np.array([h/sigma, -h*np.tan(np.radians(theta))*np.pi/180,
                     -h/d, -h/rho])
assert np.allclose(grad, grad_ref, rtol=1e-7, atol=1e-12)
d_ref = 4*sigma*np.cos(np.radians(theta))/(rho*G*h_cilj)
assert abs(d_star-d_ref) <= sirina[-1]/2 + 1e-15
assert np.isclose(1000*h,17.9987211521,atol=1e-9,rtol=0)
assert kapilarni_uspon(sigma,110,d,rho) < 0


## Granica modela

Jednadžba pretpostavlja kružnu kapilaru, statičku ravnotežu, poznat kontaktni
kut i zanemarivu gravitacijsku promjenu zakrivljenosti meniska. Kod vrlo sitnih
ili onečišćenih cijevi histereza kontaktnog kuta može dominirati rezultatom.


## 4. Dva procjepa i dvije sile — Z3

Ploča se giba udesno. Oba viskozna otpora na njoj usmjerena su ulijevo;
svaki procjep ima vlastitu promjenu brzine od nule do brzine ploče.

In [ ]:
mu, area, speed = .12, .020, .30
gaps = np.array([1e-3,2e-3])
stress = mu*speed/gaps
drag_x = -area*stress
pull_x = -drag_x.sum()
print('Naprezanja [Pa]:',stress,'; sile ulja [N]:',drag_x,'; vuča [N]:',pull_x)
assert np.allclose(drag_x,[-.72,-.36],rtol=1e-12)
assert np.isclose(pull_x,1.08,rtol=1e-12)
assert np.isclose(pull_x*speed, np.sum(mu*(speed/gaps)**2*area*gaps),rtol=1e-12)

## 5. Model mora slijediti podatke — Z5

Sile su sintetički nastavni podatci pri istoj temperaturi. Ne fitiramo
jedinstven nenewtonski zakon iz samo tri retka: provjeravamo je li stalan
omjer naprezanja i gradijenta uopće prihvatljiv.

In [ ]:
speed_data=np.array([.10,.20,.40])
force_a=np.array([.20,.40,.80]); force_b=np.array([.30,.45,.60])
area, gap = .010,1e-3
rates=speed_data/gap
tau_a,tau_b=force_a/area,force_b/area
mu_a,mu_b=tau_a/rates,tau_b/rates
constant=lambda values:np.allclose(values,values[0],rtol=1e-12)
assert constant(mu_a) and not constant(mu_b)
assert np.allclose(mu_b,[.30,.225,.15],rtol=1e-12)
prediction=mu_a[0]*area*.30/gap
bad_prediction=mu_b[0]*area*speed_data[-1]/gap
assert np.isclose(prediction,.60) and np.isclose(bad_prediction,1.20)
assert not constant(np.array([.20,.40,.70])/area/rates)
print('A: μ =',mu_a,'; B: μ_prividna =',mu_b)
print('F_A(0,30 m/s) =',prediction,'N; B iz prve točke:',bad_prediction,'N umjesto 0,60 N')
fig,ax=plt.subplots(figsize=(6.8,3.8))
ax.plot(rates,tau_a,'o-',label='A')
ax.plot(rates,tau_b,'s-',label='B; spojnice samo vode pogled')
ax.plot(rates,mu_b[0]*rates,'--',label='B: pogrešna ekstrapolacija prve točke')
ax.set(xlabel='gradijent brzine (1/s)',ylabel='naprezanje (Pa)')
ax.legend();ax.grid(ls=':');plt.show()

## 6. Odvojena stanja kapilare — P4 i Z6

Konkavni meniskus pri punjenju snižava tlak vode prema okolini. Kada je
igla puna i postoji izlazna kapljica, taj se meniskus više ne oduzima.
Koristimo isti idealizirani statički model kao u tekstu: zanemarena težina
kapljice i gubitci, kontaktna linija zadržana na rubu. Promjeri u Z6 čine
zadani interval, a ne standardnu nesigurnost.

In [ ]:
sigma_water,rho_water=.072,998.
def states(d,D,H):
    suction=4*sigma_water/d
    filling=max(0.,rho_water*G*H-suction)
    formed=rho_water*G*H+4*sigma_water/np.asarray(D)
    return filling,formed
p_fill,p_drop=states(.80e-3,2.4e-3,.060)
assert np.isclose(p_fill,227.4228,atol=1e-9,rtol=0)
assert np.isclose(p_drop,707.4228,atol=1e-9,rtol=0)
assert states(1.6e-3,2.4e-3,.060)[0] > p_fill
assert states(1.6e-3,2.4e-3,.060)[1] == p_drop
print(f'P4: punjenje {p_fill:.3f} Pa; formirana kapljica {p_drop:.3f} Pa')

diameters=np.linspace(1.6e-3,2.0e-3,101)
p_start,pressures=states(.50e-3,diameters,.042)
nominal=states(.50e-3,1.8e-3,.042)[1]
reserve=600-max(p_start,pressures.max())
omitted_head=rho_water*G*diameters[-1]
assert p_start==0 and np.isclose(nominal,571.19596,atol=1e-8,rtol=0)
assert np.all(np.diff(pressures)<0)
assert 500 < pressures.max() < 600
assert np.isclose(reserve,8.80404,atol=1e-8,rtol=0)
assert omitted_head > reserve
print(f'Z6: {pressures.min():.3f}–{pressures.max():.3f} Pa; rezerva {reserve:.3f} Pa')
print(f'Skala zanemarenog tlaka kroz kapljicu: {omitted_head:.3f} Pa')
fig,ax=plt.subplots(figsize=(6.8,3.8))
ax.plot(diameters*1000,pressures,label='statički model formirane kapljice')
ax.axhline(600,color='green',ls='--',label='gornja granica regulatora')
ax.axhline(500,color='red',ls='--',label='drugi regulator')
ax.set(xlabel='promjer kapljice (mm)',ylabel='pretlak spremnika (Pa)')
ax.legend();ax.grid(ls=':');plt.show()

## 7. Protumači

1. Zašto dva procjepa u Z3 ne smijemo zamijeniti njihovim zbrojem?
2. Što dodatna mjerna točka može promijeniti u odluci o newtonskom modelu?
3. Zašto tri točke uzorka B ne određuju jedinstven konstitutivni zakon?
4. Koji ulaz najviše pridonosi standardnoj nesigurnosti kapilarnog uspona?
5. Zašto promjer igle utječe na punjenje, ali ne na tlak već formirane
   kapljice zadanog promjera u ovom statičkom modelu?
6. Koji kraj intervala promjera kapljice daje najveći potrebni tlak?
7. Zašto pozitivan ostatak raspona regulatora od 8,8 Pa ne dokazuje da će
   stvarni uređaj raditi? Je li procjena ρgD točna korekcija tlaka ili tek
   upozorenje da zanemareni učinak može biti važan?
8. Koje dodatne podatke treba prikupiti za rast, odvajanje i protok kapljica?